In [0]:
# ================================================
# NOTEBOOK 1 — Bronze Ingest
# Connect to ADLS and read CSV files
# ================================================

storage_account = "ecommerceflowstorage2"
container = "bronze"
sas_token = "sv=2026-02-06&ss=b&srt=co&sp=rwdlacyx&se=2026-12-31T06:54:05Z&st=2026-06-13T21:39:05Z&spr=https&sig=nopl%2BhBvkatAZtw0mhQPOohEo1vAQ2lRnJ955O%2BoHVc%3D"

# Configure ADLS connection
spark.conf.set(
    f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net",
    sas_token
)

# Read orders CSV
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"wasbs://{container}@{storage_account}.blob.core.windows.net/olist_orders_dataset.csv")

print(f"✅ Orders loaded: {df_orders.count()} rows")
df_orders.show(3)

✅ Orders loaded: 99441 rows
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c

In [0]:
# Load all 9 CSV files from Bronze
base_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net"

df_customers = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_customers_dataset.csv")
df_products  = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_products_dataset.csv")
df_sellers   = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_sellers_dataset.csv")
df_items     = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_order_items_dataset.csv")
df_payments  = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_order_payments_dataset.csv")
df_reviews   = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_order_reviews_dataset.csv")
df_geo       = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/olist_geolocation_dataset.csv")
df_category  = spark.read.option("header","true").option("inferSchema","true").csv(f"{base_path}/product_category_name_translation.csv")

# Print row counts
print("=" * 45)
print(f"{'TABLE':<15} {'ROWS':>10}  {'COLS':>6}")
print("=" * 45)
for name, df in {
    'orders'   : df_orders,
    'customers': df_customers,
    'products' : df_products,
    'sellers'  : df_sellers,
    'items'    : df_items,
    'payments' : df_payments,
    'reviews'  : df_reviews,
    'geo'      : df_geo
}.items():
    print(f"{name:<15} {df.count():>10,}  {len(df.columns):>6}")
print("=" * 45)
print("✅ All tables loaded successfully!")

TABLE                 ROWS    COLS
orders              99,441       8
customers           99,441       5
products            32,951       9
sellers              3,095       4
items              112,650       7
payments           103,886       5
reviews            104,162       7
geo              1,000,163       5
✅ All tables loaded successfully!


In [0]:
print("DATA QUALITY REPORT — BRONZE LAYER")
print("=" * 50)

tables = {
    'orders'   : df_orders,
    'customers': df_customers,
    'products' : df_products,
    'sellers'  : df_sellers,
    'items'    : df_items,
    'payments' : df_payments,
    'reviews'  : df_reviews,
    'geo'      : df_geo,
    'category' : df_category
}

for name, df in tables.items():
    total_rows = df.count()
    
    total_nulls = 0
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        total_nulls += null_count
    
    duplicates = total_rows - df.dropDuplicates().count()
    
    print(f"\n📋 {name.upper()}")
    print(f"   Total rows  : {total_rows:,}")
    print(f"   Total nulls : {total_nulls:,}")
    print(f"   Duplicates  : {duplicates:,}")
    
    if total_nulls == 0 and duplicates == 0:
        print(f"   Status      : ✅ CLEAN")
    else:
        print(f"   Status      : ⚠️  NEEDS ATTENTION")

print("\n" + "=" * 50)
print("✅ Data quality check complete!")

DATA QUALITY REPORT — BRONZE LAYER

📋 ORDERS
   Total rows  : 99,441
   Total nulls : 4,908
   Duplicates  : 0
   Status      : ⚠️  NEEDS ATTENTION

📋 CUSTOMERS
   Total rows  : 99,441
   Total nulls : 0
   Duplicates  : 0
   Status      : ✅ CLEAN

📋 PRODUCTS
   Total rows  : 32,951
   Total nulls : 2,448
   Duplicates  : 0
   Status      : ⚠️  NEEDS ATTENTION

📋 SELLERS
   Total rows  : 3,095
   Total nulls : 0
   Duplicates  : 0
   Status      : ✅ CLEAN

📋 ITEMS
   Total rows  : 112,650
   Total nulls : 0
   Duplicates  : 0
   Status      : ✅ CLEAN

📋 PAYMENTS
   Total rows  : 103,886
   Total nulls : 0
   Duplicates  : 0
   Status      : ✅ CLEAN

📋 REVIEWS
   Total rows  : 104,162
   Total nulls : 177,402
   Duplicates  : 85
   Status      : ⚠️  NEEDS ATTENTION

📋 GEO
   Total rows  : 1,000,163
   Total nulls : 0
   Duplicates  : 261,831
   Status      : ⚠️  NEEDS ATTENTION

📋 CATEGORY
   Total rows  : 71
   Total nulls : 0
   Duplicates  : 0
   Status      : ✅ CLEAN

✅ Data quality

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

print("=" * 55)
print("AUDIT LOG — BRONZE INGEST")
print("=" * 55)
print(f"Run timestamp   : {datetime.now()}")
print(f"Storage account : {storage_account}")
print(f"Container       : {container}")
print(f"Pipeline        : Bronze Ingest")
print("=" * 55)

print(f"\n{'TABLE':<15} {'ROWS':>10} {'NULLS':>10} {'DUPS':>10} {'STATUS'}")
print("-" * 60)

tables = {
    'orders'   : df_orders,
    'customers': df_customers,
    'products' : df_products,
    'sellers'  : df_sellers,
    'items'    : df_items,
    'payments' : df_payments,
    'reviews'  : df_reviews,
    'geo'      : df_geo,
    'category' : df_category
}

total_rows_all  = 0
total_nulls_all = 0
total_dups_all  = 0
clean_count     = 0
attention_count = 0

for name, df in tables.items():
    total_rows = df.count()
    
    # Count nulls
    total_nulls = 0
    for c in df.columns:
        null_count = df.filter(F.col(c).isNull()).count()
        total_nulls += null_count
    
    # Count duplicates
    duplicates = total_rows - df.dropDuplicates().count()
    
    # Status
    if total_nulls == 0 and duplicates == 0:
        status = "✅ CLEAN"
        clean_count += 1
    else:
        status = "⚠️ ATTENTION"
        attention_count += 1
    
    total_rows_all  += total_rows
    total_nulls_all += total_nulls
    total_dups_all  += duplicates
    
    print(f"{name:<15} {total_rows:>10,} {total_nulls:>10,} {duplicates:>10,} {status}")

print("-" * 60)
print(f"{'TOTAL':<15} {total_rows_all:>10,} {total_nulls_all:>10,} {total_dups_all:>10,}")
print("=" * 55)

print(f"\n📋 AUDIT SUMMARY:")
print(f"   Total tables loaded  : {len(tables)}")
print(f"   Total rows ingested  : {total_rows_all:,}")
print(f"   Total nulls found    : {total_nulls_all:,}")
print(f"   Total duplicates     : {total_dups_all:,}")
print(f"   Clean tables         : {clean_count}")
print(f"   Tables needing fix   : {attention_count}")
print(f"   Pipeline end time    : {datetime.now()}")
print(f"   Pipeline status      : ✅ BRONZE INGEST COMPLETE!")
print("=" * 55)

AUDIT LOG — BRONZE INGEST
Run timestamp   : 2026-06-13 22:59:32.584763
Storage account : ecommerceflowstorage2
Container       : bronze
Pipeline        : Bronze Ingest

TABLE                 ROWS      NULLS       DUPS STATUS
------------------------------------------------------------
orders              99,441      4,908          0 ⚠️ ATTENTION
customers           99,441          0          0 ✅ CLEAN
products            32,951      2,448          0 ⚠️ ATTENTION
sellers              3,095          0          0 ✅ CLEAN
items              112,650          0          0 ✅ CLEAN
payments           103,886          0          0 ✅ CLEAN
reviews            104,162    177,402         85 ⚠️ ATTENTION
geo              1,000,163          0    261,831 ⚠️ ATTENTION
category                71          0          0 ✅ CLEAN
------------------------------------------------------------
TOTAL            1,555,860    184,758    261,916

📋 AUDIT SUMMARY:
   Total tables loaded  : 9
   Total rows ingested  :